In [1]:
#!/usr/bin/env python

# make sure to install these packages before running:
# pip install pandas
# pip install sodapy
import pandas as pd
from sodapy import Socrata
import numpy as np

# Unauthenticated client only works with public data sets. Note 'None'
# in place of application token, and no username or password:
client = Socrata("data.transportation.gov", None)

client = Socrata( "data.transportation.gov",
                     '1hCte9vYL7IpE7pFTGR1cbn8y',
                     username="hugo.baca4515@coyote.csusb.edu",
                     password="Hugobacaalex1!" )

# First 2000 results, returned as JSON from API / converted to Python list of
# dictionaries by sodapy.

results = client.get("m2f8-22s6" , where="revisiondate between '2023-01-01' and '2023-10-15'")


# Convert to pandas DataFrame
results_df_2023 = pd.DataFrame.from_records(results)

In [2]:
results_df_2023.columns.values.tolist()

['revisiondate',
 'railroadcode',
 'railroadname',
 'reportingagencytypecode',
 'reportingagencytype',
 'reasoncode',
 'reasondescription',
 'crossingid',
 'crossingidsuffix',
 'reportingagencycode',
 'reportingagencyname',
 'statecode',
 'statename',
 'countycode',
 'countyname',
 'innearcode',
 'innear',
 'citycode',
 'cityname',
 'citydescription',
 'street',
 'highwayname',
 'separatetrack',
 'sametrack',
 'railroaddivision',
 'railroadsubdivision',
 'railroadmilepostnumber',
 'timetablestation',
 'parentrailroadcode',
 'crossingownercode',
 'crossingtypecode',
 'crossingtype',
 'crossingpurposecode',
 'crossingpurpose',
 'crossingpositioncode',
 'crossingposition',
 'publicaccess',
 'typeoftrainserviceids',
 'typeoftrainserviceid1',
 'typeoftrainservice1',
 'ltpassengertrainperday',
 'numberpassengertrainperday',
 'developmenttypecode',
 'developmenttype',
 'adjacentcrossing',
 'whistlebancode',
 'whistleban',
 'hsrcorridorid',
 'latitude',
 'longitude',
 'latlongsourcecode',
 'la

In [3]:
results_df_2023.head()


,revisiondate,railroadcode,railroadname,reportingagencytypecode,reportingagencytype,reasoncode,reasondescription,crossingid,crossingidsuffix,reportingagencycode,...,advancewarning5,advancewarningtext5,advancewarning6,advancewarningtext6,ccmqzid,crossingsurfaceid3,crossingsurfacedescription3,separatetrackrailroad2,advancewarning7,advancewarningtext7
0,2023-01-02T00:00:00.000,UP,Union Pacific Railroad Company,1,Railroad,20,Date Change Only,448211M,UP,UP,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2023-01-02T00:00:00.000,UP,Union Pacific Railroad Company,1,Railroad,20,Date Change Only,253549F,UP,UP,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2023-01-03T00:00:00.000,DME,"Dakota, Minnesota & Eastern Railroad",1,Railroad,14,Change in Data,375696F,DME,DME,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2023-01-03T00:00:00.000,DME,"Dakota, Minnesota & Eastern Railroad",1,Railroad,14,Change in Data,689585H,DME,DME,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2023-01-03T00:00:00.000,DME,"Dakota, Minnesota & Eastern Railroad",1,Railroad,14,Change in Data,375636W,DME,DME,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
results_df_2023.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Columns: 246 entries, revisiondate to advancewarningtext7
dtypes: object(246)
memory usage: 1.9+ MB


In [5]:
results_df_2023['annualaveragedailytrafficcount'].value_counts()

1        109
10        21
50        14
25        14
20        12
        ... 
9926       1
5279       1
4920       1
1050       1
11400      1
Name: annualaveragedailytrafficcount, Length: 375, dtype: int64

So we should clean up the result data fram

In [6]:
#Explore for NA's, missing data, and redunnacy

isnaValues = results_df_2023.isna()
print(isnaValues)

# Check for NA values
missing_values = results_df_2023.isna().sum()

#check for null values
missing_data = results_df_2023.isnull().sum()
print(missing_data)

# Check for missing data
for column in results_df_2023.columns:
    if missing_values[column] > 0:
        print(f"The column {column} has {missing_values[column]} missing values.")

# Check for redundancy
for column in results_df_2023.columns:
    if results_df_2023[column].duplicated().sum() > 0:
        print(f"The column {column} has {results_df_2023[column].duplicated().sum()} duplicate values.")

#lets try to clean up raw data

# Calculate the percentage of missing values in each column
missing_values = results_df_2023.isna().mean()

# Drop columns with more than 30% missing values
drop_list = [column for column in missing_values.index if missing_values[column] > 0.3]

# Drop the columns
crossing_inventory_data_clean = results_df_2023.drop(columns=drop_list)


# Check the shape of the raw data
print(f"The Raw Data Frame has {results_df_2023.shape[0]} rows and {results_df_2023.shape[1]} columns.")
print(f"\n The Raw Data Frame has {len(results_df_2023.columns)} labels.")
print(f"\n The Clean Data Frame now has {len(crossing_inventory_data_clean.columns)} labels.")
print("\n Count of labels dropped: \n", len(results_df_2023.columns) - len(crossing_inventory_data_clean.columns) )
print("\n Deleted Labels: \n", drop_list)
print("\n Cleaned Data Labels: \n", crossing_inventory_data_clean.columns.values)

     revisiondate  railroadcode  railroadname  reportingagencytypecode  \
0           False         False         False                    False   
1           False         False         False                    False   
2           False         False         False                    False   
3           False         False         False                    False   
4           False         False         False                    False   
..            ...           ...           ...                      ...   
995         False         False         False                    False   
996         False         False         False                    False   
997         False         False         False                    False   
998         False         False         False                    False   
999         False         False         False                    False   

     reportingagencytype  reasoncode  reasondescription  crossingid  \
0                  False       False    

In [17]:

from tabulate import tabulate
print(tabulate(crossing_inventory_data_clean, headers = 'keys', tablefmt = 'psql'))

+-----+-------------------------+----------------+----------------------------------------------+---------------------------+-----------------------+--------------+--------------------------------+--------------+--------------------+-----------------------+----------------------------------------------+-------------+----------------+--------------+------------------+--------------+----------+------------+-----------------+------------------------------------------------------------+----------------------------------+---------------------------+-----------------+-------------+---------------------+-----------------------+--------------------------+--------------------+----------------------+---------------------+--------------------+----------------+-----------------------+-------------------+------------------------+--------------------+-------------------------+-------------------------+-----------------------+--------------------------+------------------------------+-----------------

There are a few key elements in this data set, such as 
Highway Traffic Signal Preemption
Highway Traffic Signal Preemption Code
Highway Traffic Pre Signals 
Annual Average Daily Traffic Count

Altough, some variables are missing more than 30% so they are not present in our 'clean' data set. We will pull those from our raw data set. 

In [7]:
results_df_2023['hwytrafficsgnlpre'].value_counts()

Simultaneous    24
Advanced         6
Name: hwytrafficsgnlpre, dtype: int64

In [8]:
results_df_2023['highwaytrafficpresignals']

0       No
1       No
2       No
3      NaN
4       No
      ... 
995     No
996     No
997    NaN
998     No
999     No
Name: highwaytrafficpresignals, Length: 1000, dtype: object

In [9]:
results_df_2023['hwytrafficsgnlinterconn']

0                      NaN
1       Not Interconnected
2       Not Interconnected
3                      NaN
4       Not Interconnected
              ...         
995     Not Interconnected
996     Not Interconnected
997     Not Interconnected
998     Not Interconnected
999    For Traffic Signals
Name: hwytrafficsgnlinterconn, Length: 1000, dtype: object

In [10]:
results_df_2023['hwytrafficsgnlinterconncode']

0      NaN
1        1
2        1
3      NaN
4        1
      ... 
995      1
996      1
997      1
998      1
999      2
Name: hwytrafficsgnlinterconncode, Length: 1000, dtype: object

In [11]:
results_df_2023['hwytrafficsgnlprecode']

0      NaN
1      NaN
2      NaN
3      NaN
4      NaN
      ... 
995    NaN
996    NaN
997    NaN
998    NaN
999      1
Name: hwytrafficsgnlprecode, Length: 1000, dtype: object

In [12]:
results_df_2023['hwytrafficsgnlpre']

0               NaN
1               NaN
2               NaN
3               NaN
4               NaN
           ...     
995             NaN
996             NaN
997             NaN
998             NaN
999    Simultaneous
Name: hwytrafficsgnlpre, Length: 1000, dtype: object

I have a list of crossing id's where the most accidents happen, I now want to get a list of the most trafficed crossings

In [13]:
subset_df = results_df_2023[['crossingid', 'annualaveragedailytrafficcount']]
sorted_subset_df = subset_df.sort_values(by='annualaveragedailytrafficcount', ascending=False)
print(sorted_subset_df)


    crossingid annualaveragedailytrafficcount
372    245283D                           9926
880    299452N                            980
612    634035D                            975
377    510935A                            966
354    510925U                            966
..         ...                            ...
984    228566G                            NaN
985    639526U                            NaN
991    228582R                            NaN
992    643034V                            NaN
994    976907K                            NaN

[1000 rows x 2 columns]


In [14]:
np.savez("../Data/Internal/processed/2023crossingidvsADD.npz", sorted_subset_df.to_numpy())

In [15]:
results_df_2023['hwytrafficsgnlpre'].value_counts() 

Simultaneous    24
Advanced         6
Name: hwytrafficsgnlpre, dtype: int64

In [16]:
# For each column in Raw_2023_clean, extract frequency of all values
for column in crossing_inventory_data_clean.columns:
    print(f"Frequency of values in column '{column}':")
    print(crossing_inventory_data_clean[column].value_counts())
    print("\n")

Frequency of values in column 'revisiondate':
2023-01-09T00:00:00.000    357
2023-01-03T00:00:00.000    240
2023-01-05T00:00:00.000    153
2023-01-06T00:00:00.000     97
2023-01-10T00:00:00.000     89
2023-01-04T00:00:00.000     42
2023-01-07T00:00:00.000     12
2023-01-08T00:00:00.000      8
2023-01-02T00:00:00.000      2
Name: revisiondate, dtype: int64


Frequency of values in column 'railroadcode':
CSX     283
WTNN    168
DME     152
WNYP     69
NS       64
VTR      57
GDLK     36
BNSF     32
AVR      30
GMRC     30
UP       15
MRL      14
CLP       9
TPW       8
SOO       7
GFRR      6
LIRC      4
YRC       2
MNA       2
GSWR      2
SCAX      2
AKMD      1
LAS       1
SCTR      1
CT        1
CRSH      1
SLRS      1
TMBL      1
KCS       1
Name: railroadcode, dtype: int64


Frequency of values in column 'railroadname':
CSX Transportation                              283
West Tennessee Railroad, LLC                    168
Dakota, Minnesota & Eastern Railroad            152
Western N

In [18]:
crossing_inventory_data_clean['crossingid']

0      448211M
1      253549F
2      375696F
3      689585H
4      375636W
        ...   
995    637730L
996    976973X
997    348400H
998    629310L
999    637722U
Name: crossingid, Length: 1000, dtype: object

In [19]:
crossing_inventory_data_clean['crossingid'].value_counts()

448211M    1
851167K    1
851304P    1
851301U    1
851274A    1
          ..
691958E    1
450379F    1
691969S    1
972583W    1
637722U    1
Name: crossingid, Length: 1000, dtype: int64

In [54]:
form54_data = np.load('../form54_2023_Clean_Data_CrossingIds_frequency.npz')

In [68]:
form54_data_0 =  form54_data['arr_0']
print(form54_data_0)



KeyError: '0 is not a file in the archive'